# 297. Serialize and Deserialize Binary Tree
**Difficulty:** 🔴 Hard · **Topic:** Trees · **Pattern:** [Build and Serialize](README.md) · **LeetCode:** https://leetcode.com/problems/serialize-and-deserialize-binary-tree/

## 💡 Concepts

**Core concept(s):** A **preorder DFS** that writes values with explicit **markers for empty children**, and reads them back in the same order.

**Why it applies here:** To rebuild a tree exactly, the text must capture shape too — including where children are missing. Writing a marker (like `#`) for every empty spot makes the preorder sequence enough to reconstruct the tree uniquely.

**Key intuition:** Write root, then left, then right — and write a placeholder for every empty child so nothing is ambiguous.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

### 📚 What is BFS / a Queue (used in Approach 1)?
A **queue** is first-in, first-out: you add at the back and remove from the front. **BFS (breadth-first search)** uses a queue to visit a tree level by level — pop a node, push its children, repeat.
- **In Python:** `collections.deque` gives O(1) `append` and `popleft` (a plain list's `pop(0)` is O(n)).
- **Complexity:** each node enters and leaves the queue once → **O(n)** time; the queue holds up to one level (the tree's width).

---

**Prerequisite knowledge:**
- Preorder traversal.
- Reading tokens back in the same order to rebuild.

## 📝 Problem

Write `serialize(root) -> str` and `deserialize(str) -> root` so a tree survives the round trip exactly.

**Example**
```
[1,2,3,None,None,4,5] -> "1,2,#,#,3,4,#,#,5,#,#" -> back to the same tree
```

> Two approaches, both `O(n)`: BFS level order with null markers (the shape LeetCode itself prints), then preorder DFS with null markers (shorter, the usual interview answer).

In [1]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — BFS Level Order with Null Markers

**Idea:** `serialize`: walk the tree level by level with a queue. Pop a node; if it is empty write `#`, otherwise write its value and enqueue **both** children (even empty ones). `deserialize`: the first token is the root. Keep a queue of parents still waiting for children; for each parent the next two tokens are its left and right child (`#` = none). Every real node is followed by exactly two child tokens, so the index never runs past the end.

**Time:** `O(n)` for both directions — each node and each empty-child marker is handled once.

**Space:** `O(n)` — the output has `2n + 1` tokens, and the queue holds at most one level plus its markers.

In [2]:
def serialize_bfs(root: Optional[TreeNode]) -> str:
    out = []
    q = deque([root])                     # level-order queue; it may hold None placeholders
    while q:
        n = q.popleft()
        if not n:
            out.append("#")               # empty child -> marker, nothing to enqueue
            continue
        out.append(str(n.val))            # record this node's value
        q.append(n.left); q.append(n.right)  # enqueue BOTH children, even if None
    return ",".join(out)                  # e.g. "1,2,3,#,#,4,5,#,#,#,#"

def deserialize_bfs(data: str) -> Optional[TreeNode]:
    vals = data.split(",")
    if vals[0] == "#":                    # empty tree
        return None
    root = TreeNode(int(vals[0]))
    q = deque([root]); i = 1              # parents waiting for children; i = next token
    while q:
        node = q.popleft()
        if vals[i] != "#":                # token i is this node's left child
            node.left = TreeNode(int(vals[i])); q.append(node.left)
        i += 1
        if vals[i] != "#":                # token i is this node's right child
            node.right = TreeNode(int(vals[i])); q.append(node.right)
        i += 1
    return root

### Approach 2 — Preorder with Null Markers (optimal)

**Idea:** `serialize`: preorder, writing each value and a `#` for each empty child. `deserialize`: read tokens in the same order; `#` means "no node", otherwise make a node and fill its left then right recursively.

**Time:** `O(n)` for both directions.

**Space:** `O(n)`.

In [3]:
def serialize(root: Optional[TreeNode]) -> str:
    out = []
    def dfs(n):                            # preorder walk, writing '#' for empty children
        if not n:
            out.append("#")               # marker records where a child is missing
            return
        out.append(str(n.val))            # record this node's value
        dfs(n.left); dfs(n.right)         # then its left subtree, then its right
    dfs(root)
    return ",".join(out)                  # e.g. "1,2,#,#,3,#,#"

def deserialize(data: str) -> Optional[TreeNode]:
    vals = iter(data.split(","))          # read tokens in the SAME order serialize wrote them
    def build():
        v = next(vals)
        if v == "#":                      # a '#' means "no node here"
            return None
        node = TreeNode(int(v))
        node.left = build()               # rebuild left subtree first (matches preorder)
        node.right = build()              # then the right subtree
        return node
    return build()

In [4]:
# Correctness check — round trip must reproduce the tree
def roundtrip(root):
    return deserialize(serialize(root))

def roundtrip_bfs(root):
    return deserialize_bfs(serialize_bfs(root))

tests = [[1,2,3,None,None,4,5], [1], [], [5,4,7,3,None,2,None,-1,None,9]]
for vals in tests:
    root = build_tree(vals)
    back = roundtrip(root)
    print(f"{vals}\n   preorder -> {serialize(root)}\n   bfs      -> {serialize_bfs(root)}")
    assert same_shape(root, back), "round trip changed the tree!"
    assert same_shape(root, roundtrip_bfs(root)), "BFS round trip changed the tree!"
print("\nAll tests passed")

[1, 2, 3, None, None, 4, 5]
   preorder -> 1,2,#,#,3,4,#,#,5,#,#
   bfs      -> 1,2,3,#,#,4,5,#,#,#,#
[1]
   preorder -> 1,#,#
   bfs      -> 1,#,#
[]
   preorder -> #
   bfs      -> #
[5, 4, 7, 3, None, 2, None, -1, None, 9]
   preorder -> 5,4,3,-1,#,#,#,#,7,2,9,#,#,#,#
   bfs      -> 5,4,7,3,#,2,#,-1,#,9,#,#,#,#,#

All tests passed


## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [5]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def roundtrip(root):
    return deserialize(serialize(root))

def roundtrip_bfs(root):
    return deserialize_bfs(serialize_bfs(root))

def make_worst_case(n):
    return (build_balanced(n),)
solutions = {
    "BFS level order O(n)": roundtrip_bfs,
    "preorder DFS    O(n)": roundtrip,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)



BFS level order O(n)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
     1000 |       0.86 |           n/a
     2000 |       0.66 |         0.77x
     4000 |       1.28 |         1.95x
     8000 |       2.42 |         1.89x

preorder DFS    O(n)
        n |  time (ms) | ratio vs prev
  ------- | ---------- | -------------
     1000 |       0.31 |           n/a
     2000 |       0.57 |         1.86x
     4000 |       1.12 |         1.96x
     8000 |       2.44 |         2.18x

Ratios per doubling: ~1x=>constant/log, ~2x=>linear or n log n, ~4x=>quadratic, ~8x=>cubic.

[plot skipped] matplotlib not installed - run: pip install matplotlib


{'BFS level order O(n)': [0.8591250516474247,
  0.6578329484909773,
  1.2821659911423922,
  2.4176250444725156],
 'preorder DFS    O(n)': [0.3079159650951624,
  0.571875018067658,
  1.1215419508516788,
  2.444749930873513]}

## 🧩 Patterns Learned

- **Pattern:** [Build and Serialize](README.md) — preorder (or level order) plus an explicit `#` for every empty child pins the shape down, so one traversal is enough to rebuild the tree.
- **Preorder + null markers = a lossless snapshot:** recording empties is what makes the shape recoverable.
- **Read back in the write order:** deserialize mirrors serialize exactly.
- **Signal:** "serialize / deserialize", "save and restore a tree", "encode a structure".
- **Related problems:** Construct from Traversals, Subtree of Another Tree, Encode and Decode Strings.
- **Common pitfalls:** (1) omitting markers for empty children (ambiguous); (2) reading tokens in a different order than they were written.